# 03 — Tạo segment 2 giây và kiểm tra preprocessing
Đầu vào: split đã freeze. Đầu ra: mảng `waveforms.npy` (mỗi hàng một event),
`segments.csv` ghi nguồn/biên/valid region và báo cáo.
Chỉ đọc WAV official train: train để audit preprocessing; validation áp dụng cùng quy tắc sau freeze.
**Không đọc test WAV.** Chưa đo ridge, chưa chọn synthesis anchor.
Các ô probe dùng tín hiệu toán học để kiểm code; không phải kiểm chứng y khoa.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, hashlib, json, platform, sys, time, uuid
import importlib.metadata as metadata
import numpy as np
from IPython.display import display, Markdown

PROJECT_PATH = ""  # Chỉ điền thư mục gốc nếu kernel mở ngoài dự án.
print("Kernel:", sys.executable, "\nCWD:", Path.cwd())
if PROJECT_PATH and sys.platform != "win32" and ":" in PROJECT_PATH:
    raise ValueError("Đang dùng kernel Linux với đường dẫn Windows. Chọn kernel DSP local.")
candidates = [Path(PROJECT_PATH)] if PROJECT_PATH else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p.resolve() for p in candidates if (p / 'requirements-dsp.txt').is_file() and (p / 'dsp/notebooks').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Điền PROJECT_PATH là thư mục chứa requirements-dsp.txt; xem README.md")
STARTED = time.perf_counter()

def read_csv(path):
    with path.open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

def write_csv(path, rows, fields=None):
    with path.open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields or list(rows[0]))
        w.writeheader()
        w.writerows(rows)

def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def verify(summary):
    for relative, expected in summary['output_sha256'].items():
        assert sha256(ROOT / relative) == expected, f"Checksum sai: {relative}"

def save_summary(path, summary, outputs):
    summary.update(created_at_utc=datetime.now(timezone.utc).isoformat(),
                   python=sys.version, executable=sys.executable, platform=platform.platform(),
                   versions={n: metadata.version(n) for n in ['numpy','scipy','matplotlib','ipykernel','nbformat']},
                   elapsed_seconds=time.perf_counter()-STARTED,
                   output_sha256={str(p.relative_to(ROOT)).replace('\\','/'): sha256(p) for p in outputs})
    path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print("Đã lưu:", path.relative_to(ROOT))

def snapshot(name, destination):
    import nbformat
    nb = nbformat.read(ROOT / 'dsp/notebooks' / name, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == 'code':
            cell.outputs, cell.execution_count = [], None
    nbformat.write(nb, destination)

## 1. Tham số có đơn vị
2 s, 8 kHz theo proposal. Ở dữ liệu hiện tại mọi WAV đều native 8 kHz mono PCM16:
không cần resample thật. Hàm resample có anti-alias được kiểm riêng bằng probe 16 kHz.
DC lấy trên vùng âm thanh thật. RMS normalization dùng toàn bộ cửa sổ 2 s sau padding,
mục tiêu 0.1, peak tối đa 0.95; gain lấy giá trị nhỏ hơn để không clipping.
Floor 1e-8 là bảo vệ số học (amplitude chuẩn hóa PCM trong [-1,1]), không phải ngưỡng sinh lý.
Chỉ freeze chính sách này sau probe và đo RMS train; không dùng validation để chọn.

In [ ]:
import wave
from math import gcd
from scipy.signal import resample_poly, welch
import matplotlib.pyplot as plt

SPLIT_ID = 'split_20261026T183844Z_a620ef'
FS = 8000                 # Hz
SECONDS = 2.0             # giây
N_SAMPLES = int(FS * SECONDS)
SILENCE_FLOOR = 1e-8      # RMS toàn segment, chỉ numerical guard
TARGET_RMS = 0.10         # amplitude toàn segment sau padding
HEADROOM = 0.95           # peak amplitude tuyệt đối
split_path = ROOT / 'dsp/outputs/reports' / SPLIT_ID / 'split_summary.json'
split_summary = json.loads(split_path.read_text(encoding='utf-8'))
verify(split_summary)
source = ROOT / 'dsp/data/manifests' / SPLIT_ID
events = read_csv(source / 'events.csv')
recordings = {r['recording_id']:r for r in read_csv(source/'recordings.csv') if r['split'] in {'train','validation'}}
raw = ROOT / 'dsp/data/raw/BioCAS2022'
by_recording = defaultdict(list)
for e in events:
    if e['split'] in {'train','validation'}: by_recording[e['recording_id']].append(e)
selected = sorted([e for e in events if e['split'] in {'train','validation'} and e['included']=='true'],
                  key=lambda e:(e['split'],e['event_id']))
assert {int(r['sample_rate_hz']) for r in recordings.values()} == {FS}
assert all(r['channels']=='1' and r['sample_width_bytes']=='2' for r in recordings.values())
print('Số event cần crop:', len(selected), Counter(e['split'] for e in selected))

## 2. Biên theo sample và context được phép
Khoảng dùng dạng `[start, end)`; ms → sample làm tròn gần nhất, nửa sample lên.
Cửa sổ dài 16000 sample, đặt tâm tại trung điểm interval. Với event ngắn, chỉ mở rộng qua
các annotation **chạm nhau** có nhãn Normal hoặc cùng lớp. Không bắc cầu qua khoảng chưa gán nhãn.
Phần còn thiếu được pad zero đúng vị trí; event dài giữ phần giữa và đánh dấu truncated.
Đây là cờ interval truncation, chưa phải cờ ridge duration-censored của tuần 2.

In [ ]:
def ms_to_sample(ms, fs):
    """Đổi mili-giây sang sample, nearest với half-up; không dùng round bankers."""
    return int(np.floor(float(ms) * fs / 1000.0 + 0.5))

def crop_bounds(event, context, total_samples, fs=FS, length=N_SAMPLES):
    """Tìm biên nguồn và biên valid trong cửa sổ, không lấy context chưa xác minh."""
    a, b = (ms_to_sample(event[k], fs) for k in ['start_ms','end_ms'])
    assert 0 <= a < b <= total_samples, 'Interval không hợp lệ sau đổi sample'
    safe_a, safe_b = a, b
    allowed = {'Normal', event['label']}
    intervals = [(ms_to_sample(r['start_ms'],fs), ms_to_sample(r['end_ms'],fs))
                 for r in context if r['source_label'] in allowed]
    # Chỉ gộp interval chạm/overlap, không bỏ qua khoảng trống dù rất ngắn.
    changed = True
    while changed:
        old = safe_a, safe_b
        for left, right in intervals:
            if left <= safe_b and right >= safe_a:
                safe_a, safe_b = min(safe_a,left), max(safe_b,right)
        changed = old != (safe_a,safe_b)
    window_start = (a + b - length) // 2
    left, right = max(0,safe_a,window_start), min(total_samples,safe_b,window_start+length)
    assert left < right and max(left,a) < min(right,b), 'Crop làm mất event'
    return dict(event_start_sample=a,event_end_sample=b,source_start_sample=left,source_end_sample=right,
                window_start_sample=window_start,valid_start_sample=left-window_start,
                valid_end_sample=right-window_start,event_truncated=(a<left or b>right))

def normalize_segment(signal, bounds):
    """Bỏ DC trong valid region rồi chuẩn RMS toàn cửa sổ; padding luôn bằng 0."""
    valid = signal[bounds['source_start_sample']:bounds['source_end_sample']].astype(float)
    dc = float(valid.mean())
    valid -= dc
    rms = float(np.sqrt(np.sum(valid**2)/N_SAMPLES))
    if rms <= SILENCE_FLOOR: return None, dict(raw_rms=rms,dc=dc,reason='zero_or_numerically_silent')
    peak = float(np.max(np.abs(valid)))
    gain = min(TARGET_RMS/rms, HEADROOM/peak)
    output = np.zeros(N_SAMPLES, dtype=np.float32)
    output[bounds['valid_start_sample']:bounds['valid_end_sample']] = valid * gain
    return output, dict(raw_rms=rms,dc=dc,gain=gain,achieved_rms=float(np.sqrt(np.mean(output.astype(float)**2))),
                        peak_limited=HEADROOM/peak < TARGET_RMS/rms,reason='')

def resample_audio(signal, original_fs, target_fs=FS):
    """Polyphase FIR chống alias khi đổi sample rate; hiện dữ liệu thật không cần đổi."""
    divisor = gcd(original_fs,target_fs)
    return resample_poly(signal, target_fs//divisor, original_fs//divisor) if original_fs!=target_fs else signal.copy()

## 3. Probe cho đơn vị, padding, context, DC và anti-alias
Ví dụ cố ý có Normal chạm event, crackle và khoảng trống. Kỳ vọng được viết bằng số cụ thể,
không tính lại bằng chính hàm cần kiểm. Tone 5 kHz ở nguồn 16 kHz phải bị giảm khi xuống 8 kHz.

In [ ]:
def annotation(start, end, label):
    return dict(start_ms=start,end_ms=end,label=label,source_label=label)

assert ms_to_sample(1000,8000)==8000 and ms_to_sample(0.0625,8000)==1
event = annotation(1000,1500,'Wheeze')
alone = crop_bounds(event,[event],24000)
assert (alone['source_start_sample'],alone['source_end_sample'])==(8000,12000)
assert (alone['valid_start_sample'],alone['valid_end_sample'])==(6000,10000)
context = [annotation(0,1000,'Normal'),event,annotation(1500,2000,'Fine Crackle')]
b = crop_bounds(event,context,24000)
assert (b['source_start_sample'],b['source_end_sample'])==(2000,12000)
gap = [annotation(0,999,'Normal'),event]
assert crop_bounds(event,gap,24000)['source_start_sample']==8000
normal = annotation(0,500,'Normal')
assert crop_bounds(normal,[normal,annotation(500,1500,'Wheeze')],16000)['source_end_sample']==4000
long = annotation(0,3000,'Normal')
assert crop_bounds(long,[long],24000)['source_start_sample']==4000
assert crop_bounds(long,[long],24000)['event_truncated']
t = np.arange(24000)/FS
probe, stats = normalize_segment(0.3+0.1*np.sin(2*np.pi*440*t),alone)
assert np.all(probe[:6000]==0) and np.all(probe[10000:]==0)
assert abs(probe[6000:10000].mean()) < 1e-7
assert abs(stats['achieved_rms']-TARGET_RMS) < 1e-7 and np.max(abs(probe))<=HEADROOM+1e-7
assert normalize_segment(np.ones(24000),alone)[0] is None
t16 = np.arange(32000)/16000
low = resample_audio(np.sin(2*np.pi*1000*t16),16000)
high = resample_audio(np.sin(2*np.pi*5000*t16),16000)
assert len(low)==16000
assert np.sqrt(np.mean(low[100:-100]**2)) > 0.69
alias_rms = float(np.sqrt(np.mean(high[100:-100]**2)))
assert alias_rms < 0.01
print('PASS crop/DC/padding; anti-alias 5 kHz RMS sau downsample:', alias_rms)

## 4. Đọc WAV có kiểm checksum
Chỉ cache một recording tại một thời điểm. SHA-256 phải khớp lần tải trước khi giải mã.
Đường dẫn và split được kiểm trước `wave.open`; không mở test kể cả nhầm ID.

In [ ]:
loaded_id, loaded_signal = None, None
accessed_recordings = set()

def load_recording(recording_id):
    global loaded_id, loaded_signal
    if recording_id == loaded_id: return loaded_signal
    r = recordings[recording_id]  # Dict không chứa test.
    assert r['split'] in {'train','validation'} and r['official_partition']=='official_train'
    path = (raw/r['audio_path']).resolve()
    assert path.parent == (raw/'train2022_wav').resolve()
    assert sha256(path)==r['audio_sha256_at_download'], 'Raw đã thay đổi: '+recording_id
    with wave.open(str(path),'rb') as f:
        assert (f.getframerate(),f.getnchannels(),f.getsampwidth())==(FS,1,2)
        signal = np.frombuffer(f.readframes(f.getnframes()),dtype='<i2').astype(float)/32768.0
    loaded_id, loaded_signal = recording_id, signal
    accessed_recordings.add(recording_id)
    return signal

train_measures = []
bandwidth = []
seen_bandwidth_subjects = {'Normal':set(),'Wheeze':set()}
for e in selected:
    if e['split']!='train': continue
    signal = load_recording(e['recording_id'])
    bounds = crop_bounds(e,by_recording[e['recording_id']],len(signal))
    segment, stats = normalize_segment(signal,bounds)
    train_measures.append(dict(event_id=e['event_id'],label=e['label'],**stats))
    seen = seen_bandwidth_subjects[e['label']]
    if segment is not None and e['subject_id'] not in seen and len(seen)<20:
        seen.add(e['subject_id'])
        a,b = bounds['valid_start_sample'],bounds['valid_end_sample']
        freq, power = welch(segment[a:b],fs=FS,nperseg=min(256,b-a))
        bandwidth.append(dict(event_id=e['event_id'],subject_id=e['subject_id'],label=e['label'],
            fraction_power_above_3500_hz=float(power[freq>=3500].sum()/max(power.sum(),1e-30))))
train_rms = np.array([r['raw_rms'] for r in train_measures])
print('RMS train min/median/max:', np.quantile(train_rms,[0,0.5,1]))
print('Train numerical-silence exclusions:', sum(r['reason']!='' for r in train_measures))
print('Native bandwidth audit:', len(bandwidth),'segments, <=20 người mỗi lớp')
display(bandwidth[:3])

## 5. Freeze preprocessing trước validation
Floor phải cách xa RMS thấp nhất của train không im lặng; nếu không, dừng để xem train,
không tự bỏ hàng loạt âm nhỏ. Headroom được kiểm bằng probe; target RMS là quy ước kỹ thuật
chung, có ghi số trường hợp bị peak-limit. Đây **không** phải calibration acceptance của generator.
Audit phổ chỉ kiểm được băng thông đã ghi đến 4 kHz; không chứng minh nội dung trên 4 kHz không tồn tại.

In [ ]:
nonzero = train_rms[train_rms>SILENCE_FLOOR]
assert len(nonzero)>0 and nonzero.min()>100*SILENCE_FLOOR, 'Cần xem lại floor bằng train'
freeze = dict(created_at_utc=datetime.now(timezone.utc).isoformat(),fs_hz=FS,seconds=SECONDS,
    silence_floor=SILENCE_FLOOR,target_rms=TARGET_RMS,headroom=HEADROOM,
    minimum_train_nonzero_rms=float(nonzero.min()),train_probe_only=True,
    rounding='nearest_half_up',crop='center; contiguous verified Normal or same-label context only',
    dc='mean of valid samples',rms='whole 2-second window including zero padding',
    resample='native 8000 Hz unchanged; polyphase anti-alias separately probed at 16000 Hz',
    generator_acceptance_calibrated=False)
print('Preprocessing freeze UTC:', freeze['created_at_utc'])

## 6. Áp dụng chung cho train và validation
Mỗi event giữ đúng một hàng nếu vượt numerical guard. Không dùng ridge để lọc tập đánh giá.
`row_index` nối CSV với mảng waveform, `event_id` nối về annotation ban đầu.
`source_start/end_sample` chỉ vị trí trong WAV gốc; `valid_start/end_sample` chỉ vị trí trong segment.

In [ ]:
SEGMENT_ID = 'segments_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
out = ROOT/'dsp/data/processed'/SEGMENT_ID
report = ROOT/'dsp/outputs/reports'/SEGMENT_ID
out.mkdir(parents=True,exist_ok=False)
report.mkdir(parents=True,exist_ok=False)
waveforms, segment_rows, excluded = [], [], []
for e in selected:
    signal = load_recording(e['recording_id'])
    bounds = crop_bounds(e,by_recording[e['recording_id']],len(signal))
    x, stats = normalize_segment(signal,bounds)
    if x is None:
        excluded.append(dict(event_id=e['event_id'],split=e['split'],label=e['label'],reason=stats['reason']))
        continue
    a,b = bounds['valid_start_sample'],bounds['valid_end_sample']
    assert 0<=a<b<=N_SAMPLES and np.isfinite(x).all()
    assert np.all(x[:a]==0) and np.all(x[b:]==0) and np.max(abs(x))<=HEADROOM+1e-7
    segment_rows.append(dict(row_index=len(waveforms),event_id=e['event_id'],recording_id=e['recording_id'],
        subject_id=e['subject_id'],split=e['split'],label=e['label'],audio_path=e['audio_path'],
        annotation_path=e['annotation_path'],start_ms=e['start_ms'],end_ms=e['end_ms'],
        fs_hz=FS,**bounds,valid_fraction=(b-a)/N_SAMPLES,**stats))
    waveforms.append(x)
X = np.stack(waveforms)
del waveforms
assert len(segment_rows)+len(excluded)==len(selected)
assert len({r['event_id'] for r in segment_rows})==len(segment_rows)
assert {r['split'] for r in segment_rows}=={'train','validation'}
np.save(out/'waveforms.npy',X)
write_csv(out/'segments.csv',segment_rows)
write_csv(report/'exclusions.csv',excluded,['event_id','split','label','reason'])
write_csv(report/'train_bandwidth.csv',bandwidth)
write_csv(report/'train_amplitude.csv',train_measures)
print('Mảng:', X.shape, 'MB:', X.nbytes/1e6,'; exclusions:',len(excluded))

## 7. Audit padding, truncation và hình train
So sánh độ dài thật theo lớp trên train: khác biệt có thể tạo confound, kể cả pooling loại tâm frame
trong padding. Bảng validation chỉ mô tả sau freeze; không sửa crop/generator dựa vào bảng này.
Chọn ví dụ train theo ID, không theo điểm mô hình. Nghe bằng tai chỉ giúp hiểu tín hiệu.

In [ ]:
padding_rows = []
for part in ['train','validation']:
    for label in ['Normal','Wheeze']:
        rows = [r for r in segment_rows if r['split']==part and r['label']==label]
        fractions = np.array([r['valid_fraction'] for r in rows])
        padding_rows.append(dict(split=part,label=label,segments=len(rows),subjects=len({r['subject_id'] for r in rows}),
            padded=int(sum(v<1 for v in fractions)),median_valid_fraction=float(np.median(fractions)),
            mean_valid_fraction=float(fractions.mean()),truncated=sum(r['event_truncated'] for r in rows),
            peak_limited=sum(r['peak_limited'] for r in rows)))
display(padding_rows)
write_csv(report/'padding_counts.csv',padding_rows)
fig, axes = plt.subplots(2,2,figsize=(11,6),layout='constrained')
for i,label in enumerate(['Normal','Wheeze']):
    rows = [r for r in segment_rows if r['split']=='train' and r['label']==label]
    r = rows[0]
    axes[i,0].plot(np.arange(N_SAMPLES)/FS,X[r['row_index']],lw=.5)
    axes[i,0].axvspan(r['valid_start_sample']/FS,r['valid_end_sample']/FS,color='green',alpha=.15)
    axes[i,0].set(title=label+' | '+r['event_id'],xlabel='Time (s)',ylabel='Amplitude')
    axes[i,1].hist([q['valid_fraction'] for q in rows],bins=np.linspace(0,1,21))
    axes[i,1].set(title=label+' train: valid fraction',xlabel='Valid / 2 seconds',ylabel='Events')
fig.savefig(report/'train_padding.png',dpi=140)
plt.show()
from IPython.display import Audio
example = next(r for r in segment_rows if r['split']=='train' and r['label']=='Wheeze')
display(Audio(X[example['row_index']],rate=FS,normalize=False))

## 8. Lưu cấu hình, lineage và bằng chứng
Notebook 04 phải đọc đúng SEGMENT_ID dưới đây. Lưu mọi input hash, source snapshot và danh sách
recording đã mở để kiểm lại test gate. Không thay raw; không có synthesis ở bước này.

In [ ]:
write_csv(report/'accessed_recordings.csv',[dict(recording_id=r,split=recordings[r]['split']) for r in sorted(accessed_recordings)])
snapshot('03_tao_segment_va_kiem_tra.ipynb',report/'notebook_source.ipynb')
summary = dict(run_id=SEGMENT_ID,split_id=SPLIT_ID,split_summary_sha256=sha256(split_path),
    preprocessing=freeze,counts=padding_rows,exclusions=len(excluded),test_audio_read=False,
    crop_probe_pass=True,resample_probe_alias_rms=alias_rms,
    bandwidth_limitation='Native 8 kHz; no evidence about missing frequencies above Nyquist. Training subset only.',
    synthesis_feasibility_verified=False)
outputs = list(out.glob('*'))+list(report.glob('*.csv'))+[report/'notebook_source.ipynb',report/'train_padding.png']
save_summary(report/'segment_summary.json',summary,outputs)
(report/'BAO_CAO.md').write_text('# Segment/preprocessing\n\n'+
    str(len(segment_rows))+' segments; '+str(len(excluded))+' numerical exclusions. '+
    'Xem padding_counts.csv, train_bandwidth.csv và train_padding.png.\n\n'+
    'Probe crop/DC/padding/anti-alias PASS. Native 8 kHz không resample thật. Test chưa mở. '+
    'Chưa kiểm chứng ridge hoặc generator; padding khác nhau theo lớp cần theo dõi.\n',encoding='utf-8')
print('SEGMENT_ID =',repr(SEGMENT_ID))